## Audio Preprocessing Pipeline

```This script will read the protocol files to get the list of audio files and their labels. It will then load each audio file and convert its waveform into a spectrogram to be used as input to the LCNN model.```
```API used: tensorflow.data```

### 1.Constants and paths

In [ ]:

import tensorflow as tf
import os
import numpy as np

DATASET_BASE_DIR = 'thesis/dataset/LA' 

# Paths to protocol files and audio directories (READ_ME.txt)
TRAIN_PROTOCOL = os.path.join(DATASET_BASE_DIR, 'ASVspoof2019_LA_cm_protocols', 'ASVspoof2019.LA.cm.train.trn.txt')
DEV_PROTOCOL = os.path.join(DATASET_BASE_DIR, 'ASVspoof2019_LA_cm_protocols', 'ASVspoof2019.LA.cm.dev.trl.txt')
TRAIN_AUDIO_DIR = os.path.join(DATASET_BASE_DIR, 'ASVspoof2019_LA_train', 'flac')
DEV_AUDIO_DIR = os.path.join(DATASET_BASE_DIR, 'ASVspoof2019_LA_dev', 'flac')

# Preprocessing parameters
SAMPLE_RATE = 16000  # dataset sampling rate is 16 kHz
FIXED_DURATION_SECS = 4 # 4-second clips for a fixed input size
FIXED_LENGTH_SAMPLES = int(FIXED_DURATION_SECS * SAMPLE_RATE)

# STFT parameters for spectrogram
# FRAME_LENGTH = Typically 20-40ms, here it's 64ms
# FRAME_STEP = Typically 10-20ms overlap, here it's 32ms
FRAME_LENGTH = 1024 
FRAME_STEP = 512   
FFT_LENGTH = 1024

### 2.Parsing the protocols

```Reads a CM protocol file and returns lists of file paths and their corresponding labels.```

In [ ]:
def parse_protocol(protocol_file, audio_dir):
    file_paths = []
    labels = []
    with open(protocol_file, 'r') as f:
        for line in f:
            parts = line.strip().split(' ')
            # format is SPEAKER_ID AUDIO_FILE_NAME - SYSTEM_ID KEY  
            file_name = parts[1] + '.flac'
            key = parts[4]  # 'bonafide' or 'spoof' 

            file_paths.append(os.path.join(audio_dir, file_name))
            # Convert labels to integers: 1 for 'spoof', 0 for 'bonafide'
            labels.append(1 if key == 'spoof' else 0)

    return file_paths, labels

### 3.Audio Preprocessing

```Loads, decodes, pads/truncates, and converts a single audio file to a spectrogram.```

In [ ]:
def preprocess_path(file_path, label):
    
    # 1. Read raw binary audio file
    audio_binary = tf.io.read_file(file_path)

    # 2. Decode the .flac file to a waveform tensor 
    waveform, _ = tf.audio.decode_flac(audio_binary, desired_channels=1)
    waveform = tf.squeeze(waveform, axis=-1) # the channel dimension is removed for now (giving a grayscale image)

    # 3. Ensure all audio clips have a fixed length
    if tf.shape(waveform)[0] < FIXED_LENGTH_SAMPLES:
        # Pad with zeros if the clip is shorter
        padding = FIXED_LENGTH_SAMPLES - tf.shape(waveform)[0]
        waveform = tf.pad(waveform, [[0, padding]])
    else:
        # Truncate if the clip is longer
        waveform = waveform[:FIXED_LENGTH_SAMPLES]

    # 4. Compute the Short-Time Fourier Transform (STFT) to get a spectrogram
    stfts = tf.signal.stft(
        waveform,
        frame_length=FRAME_LENGTH,
        frame_step=FRAME_STEP,
        fft_length=FFT_LENGTH
    )
    # The result is complex numbers, so take the absolute value for magnitude
    spectrogram = tf.abs(stfts)

    # 5. Convert to a log-spectrogram for better feature representation--Will also try CQT spec and Mel-spec
    # Add a small epsilon to avoid taking log(0)
    spectrogram = tf.math.log(spectrogram + 1e-6)

    # 6. Adding a "channel" dimension for the CNN model
    spectrogram = tf.expand_dims(spectrogram, axis=-1)

    return spectrogram, label

### 4.Dataset Creation 

```A tf.data.Dataset pipeline is needed for efficient training or validation.```

In [ ]:
def create_dataset(file_paths, labels, batch_size=32):
    # Create a dataset of file paths and a dataset of labels
    path_ds = tf.data.Dataset.from_tensor_slices(file_paths)
    label_ds = tf.data.Dataset.from_tensor_slices(labels)

    # Zip the paths and labels together
    full_ds = tf.data.Dataset.zip((path_ds, label_ds))

    # Apply the preprocessing function to each element in the dataset
    full_ds = full_ds.map(preprocess_path, num_parallel_calls=tf.data.AUTOTUNE)

    # Shuffle, batch, and prefetch for optimal performance
    full_ds = full_ds.shuffle(buffer_size=1024).batch(batch_size).prefetch(buffer_size=tf.data.AUTOTUNE)

    return full_ds

### 5.Main()

In [ ]:
if __name__ == '__main__':
    print("--- Starting Audio Preprocessing Pipeline ---")

    # a) Parse protocol files to get file lists and labels
    print("Parsing protocol files...")
    train_files, train_labels = parse_protocol(TRAIN_PROTOCOL, TRAIN_AUDIO_DIR)
    dev_files, dev_labels = parse_protocol(DEV_PROTOCOL, DEV_AUDIO_DIR)
    print(f"    {len(train_files)} training files and {len(dev_files)} development files found.")

    # b) Create the TensorFlow Dataset objects
    print("\n2. Creating TensorFlow training and validation datasets...")
    # small batch size created for demonstration
    train_ds = create_dataset(train_files, train_labels, batch_size=32)
    val_ds = create_dataset(dev_files, dev_labels, batch_size=32)
    print("   Datasets created successfully.")

    # c) Demonstrate the output by inspecting one batch
    print("\n3. Inspecting one batch from the training dataset:")
    for spectrograms, labels in train_ds.take(1):
        print(f"   Spectrograms tensor shape: {spectrograms.shape}")
        print(f"   Labels tensor shape:         {labels.shape}")

    print("\n--- Pipeline Ready ---")